# Ensemble long-term blend (DLR + ESA CCI)

In [ ]:
# FIX PROJ PATH BEFORE IMPORTING GDAL

candidate_proj_dirs = [
    Path(os.environ.get("CONDA_PREFIX", "")) / "share" / "proj",
    Path(sys.prefix) / "share" / "proj",
    Path("/opt/conda/share/proj"),
    Path("/usr/share/proj"),
    Path("/usr/local/share/proj"),
]

for proj_dir in candidate_proj_dirs:
    if (proj_dir / "proj.db").exists():
        os.environ["PROJ_DATA"] = str(proj_dir)
        os.environ["PROJ_LIB"] = str(proj_dir)
        break

In [ ]:
# USER SETTINGS

MONTH_NAME = "jan"

DLR_DIR = Path("longterm_monthly")
ESACCI_DIR = Path("esacci_longterm")
OUTPUT_DIR = Path("ensemble_longterm")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NODATA = 255

HIGH_LAT_LIMIT = 30.0

# abs(latitude) > 30
HIGH_LAT_DLR_WEIGHT = 0.5
HIGH_LAT_ESACCI_WEIGHT = 0.5

# abs(latitude) <= 30
LOW_LAT_DLR_WEIGHT = 0.1
LOW_LAT_ESACCI_WEIGHT = 0.9

GDAL_THREADS = "8"
BLOCK_ROWS = 2048

gdal.UseExceptions()
gdal.SetConfigOption("GDAL_NUM_THREADS", GDAL_THREADS)

In [ ]:
# MANUAL FILE LIST FOR ONE MONTH

FILES = {
    "p05": {
        "dlr": DLR_DIR / f"snowcover_dlr.gsp.sce.{MONTH_NAME}_p05_500m_s_20000101_20251231_go_epsg.4326_v20260605.tif",
        "esacci": ESACCI_DIR / f"snowcover_esacci.{MONTH_NAME}_p05_500m_s_20000101_20231231_go_epsg.4326_v20260610.tif",
        "output": OUTPUT_DIR / f"snowcover_ensemble.{MONTH_NAME}_p05_500m_s_20000101_20251231_go_epsg.4326_v20260610.tif",
    },
    "p50": {
        "dlr": DLR_DIR / f"snowcover_dlr.gsp.sce.{MONTH_NAME}_p50_500m_s_20000101_20251231_go_epsg.4326_v20260605.tif",
        "esacci": ESACCI_DIR / f"snowcover_esacci.{MONTH_NAME}_p50_500m_s_20000101_20231231_go_epsg.4326_v20260610.tif",
        "output": OUTPUT_DIR / f"snowcover_ensemble.{MONTH_NAME}_p50_500m_s_20000101_20251231_go_epsg.4326_v20260610.tif",
    },
    "p95": {
        "dlr": DLR_DIR / f"snowcover_dlr.gsp.sce.{MONTH_NAME}_p95_500m_s_20000101_20251231_go_epsg.4326_v20260605.tif",
        "esacci": ESACCI_DIR / f"snowcover_esacci.{MONTH_NAME}_p95_500m_s_20000101_20231231_go_epsg.4326_v20260610.tif",
        "output": OUTPUT_DIR / f"snowcover_ensemble.{MONTH_NAME}_p95_500m_s_20000101_20251231_go_epsg.4326_v20260610.tif",
    },
}

In [ ]:
# Blend functions
def create_output_like(reference_ds, output_path):
    driver = gdal.GetDriverByName("GTiff")

    output_ds = driver.Create(
        str(output_path),
        reference_ds.RasterXSize,
        reference_ds.RasterYSize,
        1,
        gdal.GDT_Byte,
        options=[
            "TILED=YES",
            "BLOCKXSIZE=2048",
            "BLOCKYSIZE=2048",
            "COMPRESS=DEFLATE",
            "BIGTIFF=IF_SAFER",
            "SPARSE_OK=TRUE",
            "NUM_THREADS={}".format(GDAL_THREADS),
        ],
    )

    output_ds.SetGeoTransform(reference_ds.GetGeoTransform())
    output_ds.SetProjection(reference_ds.GetProjection())

    output_band = output_ds.GetRasterBand(1)
    output_band.SetNoDataValue(NODATA)

    return output_ds


def row_latitudes(geo_transform, y_start, n_rows):
    row_indices = y_start + np.arange(n_rows)

    latitudes = (
        geo_transform[3]
        + (row_indices + 0.5) * geo_transform[5]
    )

    return latitudes


def blend_arrays(dlr, esacci, latitudes):
    dlr_valid = dlr != NODATA
    esacci_valid = esacci != NODATA

    both_valid = dlr_valid & esacci_valid
    only_dlr = dlr_valid & ~esacci_valid
    only_esacci = ~dlr_valid & esacci_valid

    output = np.full(dlr.shape, NODATA, dtype=np.uint8)

    output[only_dlr] = dlr[only_dlr]
    output[only_esacci] = esacci[only_esacci]

    high_lat_rows = np.abs(latitudes) > HIGH_LAT_LIMIT
    high_lat_mask = high_lat_rows[:, None]
    low_lat_mask = ~high_lat_mask

    high_lat_both = both_valid & high_lat_mask
    low_lat_both = both_valid & low_lat_mask

    blended = np.empty(dlr.shape, dtype=np.float32)

    blended[high_lat_both] = (
        HIGH_LAT_DLR_WEIGHT * dlr[high_lat_both].astype(np.float32)
        + HIGH_LAT_ESACCI_WEIGHT * esacci[high_lat_both].astype(np.float32)
    )

    blended[low_lat_both] = (
        LOW_LAT_DLR_WEIGHT * dlr[low_lat_both].astype(np.float32)
        + LOW_LAT_ESACCI_WEIGHT * esacci[low_lat_both].astype(np.float32)
    )

    output[both_valid] = np.rint(blended[both_valid]).astype(np.uint8)

    return output


def blend_one_file(label, dlr_path, esacci_path, output_path):
    print("=" * 80, flush=True)
    print("Percentile: {}".format(label), flush=True)
    print("DLR:        {}".format(dlr_path), flush=True)
    print("ESACCI:     {}".format(esacci_path), flush=True)
    print("Output:     {}".format(output_path), flush=True)

    dlr_ds = gdal.Open(str(dlr_path), gdal.GA_ReadOnly)
    esacci_ds = gdal.Open(str(esacci_path), gdal.GA_ReadOnly)

    output_ds = create_output_like(esacci_ds, output_path)

    dlr_band = dlr_ds.GetRasterBand(1)
    esacci_band = esacci_ds.GetRasterBand(1)
    output_band = output_ds.GetRasterBand(1)

    geo_transform = esacci_ds.GetGeoTransform()

    width = esacci_ds.RasterXSize
    height = esacci_ds.RasterYSize

    for y_start in range(0, height, BLOCK_ROWS):
        n_rows = min(BLOCK_ROWS, height - y_start)

        dlr = dlr_band.ReadAsArray(0, y_start, width, n_rows)
        esacci = esacci_band.ReadAsArray(0, y_start, width, n_rows)

        latitudes = row_latitudes(
            geo_transform=geo_transform,
            y_start=y_start,
            n_rows=n_rows,
        )

        output = blend_arrays(
            dlr=dlr,
            esacci=esacci,
            latitudes=latitudes,
        )

        output_band.WriteArray(output, 0, y_start)

        print(
            "  Rows {}–{} done".format(
                y_start,
                y_start + n_rows - 1,
            ),
            flush=True,
        )

    output_ds.FlushCache()

    dlr_ds = None
    esacci_ds = None
    output_ds = None

    print("Finished: {}".format(output_path), flush=True)

In [ ]:
def main():
    for label, paths in FILES.items():
        blend_one_file(
            label=label,
            dlr_path=paths["dlr"],
            esacci_path=paths["esacci"],
            output_path=paths["output"],
        )

    print(f"\nEnsemble for {MONTH_NAME} completed.", flush=True)

In [ ]:
main()